# Performance Optimization

In [1]:
import pandas as pd
import numpy as np

In [2]:
sales = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 104, 105, 106],
    "Age": [22, 35, 28, 42, 31, 25],
    "Department": ["AI", "ML", "AI", "DS", "ML", "AI"],
    "Sales": [5000, 7500, 6200, 8200, 4500, 6800],
    "Quantity": [2, 3, 2, 4, 1, 3]
})

print(sales)


   Customer_ID  Age Department  Sales  Quantity
0          101   22         AI   5000         2
1          102   35         ML   7500         3
2          103   28         AI   6200         2
3          104   42         DS   8200         4
4          105   31         ML   4500         1
5          106   25         AI   6800         3


### What is Performance Optimization?
Performance optimization means improving code so that it:

* Runs faster
* Uses less memory
* Handles larger datasets efficiently
* Avoids unnecessary operations
* Scales better

### 1. Vectorization

Vectorization means performing an operation on an entire Pandas Series or NumPy array at once instead of processing individual rows one by one.

Instead of:

Row 1 → calculate

Row 2 → calculate

Row 3 → calculate
...

vectorization allows Pandas/NumPy to perform the operation efficiently on the whole column.

Example:

Suppose we want to calculate total revenue:

Revenue = Sales × Quantity

In [4]:
sales["Revenue"] = (
    sales["Sales"] * sales["Quantity"])

print(sales)

   Customer_ID  Age Department  Sales  Quantity  Revenue
0          101   22         AI   5000         2    10000
1          102   35         ML   7500         3    22500
2          103   28         AI   6200         2    12400
3          104   42         DS   8200         4    32800
4          105   31         ML   4500         1     4500
5          106   25         AI   6800         3    20400


#### Conditional Vectorization

Suppose customers with sales above 7000 get a 10% bonus.

In [5]:
sales["Bonus"] = np.where(
    sales["Sales"] > 7000,
    sales["Sales"] * 0.10,
    0
)

print(sales[[
    "Sales",
    "Bonus"
]])

   Sales  Bonus
0   5000    0.0
1   7500  750.0
2   6200    0.0
3   8200  820.0
4   4500    0.0
5   6800    0.0


### 2. apply()

apply() applies a function to values in a Series or along rows/columns of a DataFrame.

It is useful when the required transformation is more complicated than a simple vectorized operation.

In [6]:
# categorize customers based on age.
def age_group(age):
    if age < 25:
        return "Young"
    elif age < 40:
        return "Adult"
    else:
        return "Senior"

In [7]:
sales["Age_Group"] = sales["Age"].apply(
    age_group
)

print(sales[[
    "Age",
    "Age_Group"
]])

   Age Age_Group
0   22     Young
1   35     Adult
2   28     Adult
3   42    Senior
4   31     Adult
5   25     Adult


### apply() vs Loops

Both loops and apply() can process rows individually, but they differ in syntax and typical use.

In [8]:
# Loop

result = []

for value in sales["Sales"]:
    result.append(value * 2)

sales["Result"] = result

In [9]:
# apply()

sales["Result"] = sales["Sales"].apply(
    lambda x: x * 2
)

#### apply()
* Usually slower
* High readability
* Used in Custom transformations

#### Loop
* Usually slowest
* Medium readability
* Used in Complex procedural logic

### 3. Memory Optimization

Memory optimization means reducing the amount of RAM required to store and process a DataFrame.

This becomes important when working with large datasets.

For example:

Small dataset --> 1 MB

Large dataset --> 5 GB

Very large dataset --> 500 GB+

Efficient memory management becomes critical as the data grows.

In [10]:
memory = sales.memory_usage(
    deep=True
).sum()

print(memory)

607


#### Integer Optimization

Suppose a column contains:

0 to 100 --> but Pandas stores it as int64.

We may be able to use a smaller integer type.

Since age values fit easily within int8.

#### Float Optimization

Suppose a floating-point column doesn't require high precision.

We can use:

sales["Sales"] = sales["Sales"].astype("float32")

instead of float64.

For very large datasets, this can reduce memory usage.

### 4. Category Data Type

The category data type is useful for columns containing a limited number of repeated values.

For example:

Department

AI

ML

AI

DS

ML

AI

There may be millions of rows but only a few unique departments.

Instead of storing every string independently, Pandas can represent the values more efficiently using categorical encoding.

#### Convert Object to Category

In [12]:
print(sales["Department"].dtype)

str


In [14]:
sales["Department"] = sales["Department"].astype("category")
print(sales["Department"].dtype)

category


#### When Should You Use category?

Category is particularly useful when a column has:

* Repeated values
* A relatively small number of unique values
* Categorical information

Examples:

* Gender
* Department
* City
* Country
* Product_Category
* Customer_Type
* Payment_Method

### 5. Efficient Pandas Practice
 
#### Select Only Required Columns

Suppose a dataset has 50 columns but we need only:

* Customer_ID
* Sales
* Department

In [16]:
selected = sales[
    [
        "Customer_ID",
        "Sales",
        "Department"
    ]
]

print(selected)

   Customer_ID  Sales Department
0          101   5000         AI
1          102   7500         ML
2          103   6200         AI
3          104   8200         DS
4          105   4500         ML
5          106   6800         AI


#### Specify Data Types

When reading a CSV, you can specify appropriate types.

In [21]:
df = pd.read_csv(
    "employees.csv",
    dtype={
        "Employee_ID": "int32",
        "Department": "category"
    }
)
print(df["Department"].dtype)

category


#### Use query()


In [24]:
result = sales.query(
    "Sales > 6000"
)
print(result)

   Customer_ID  Age Department  Sales  Quantity  Revenue  Bonus Age_Group  \
1          102   35         ML   7500         3    22500  750.0     Adult   
2          103   28         AI   6200         2    12400    0.0     Adult   
3          104   42         DS   8200         4    32800  820.0    Senior   
5          106   25         AI   6800         3    20400    0.0     Adult   

   Result  
1   15000  
2   12400  
3   16400  
5   13600  


#### Use Appropriate File Formats

For large datasets, binary formats such as Parquet can be more efficient than CSV for repeated data processing.

In [25]:
df.to_parquet(
    "sales1.parquet",
    index=False
)

In [26]:
df = pd.read_parquet(
    "sales1.parquet"
)

#### Read Large Files in Chunks

If a CSV is too large to fit comfortably into memory, use chunksize.

In [27]:
for chunk in pd.read_csv(
    "students.csv",
    chunksize=2
):
    print(chunk.shape)

(2, 5)
(2, 5)
(1, 5)
